In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

class HelloAgentLLM:
    def __init__(self, model:str = None, apikey: str = None, baseurl: str = None, timeout:int = None):
        self.model = model or os.getenv("LLM_MODEL_ID")
        apikey = apikey or os.getenv("DEEPSEEK_API_KEY")
        baseurl = baseurl or os.getenv("DEEPSEEK_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))

        if not all([self.model, apikey, baseurl]):
            raise ValueError("模型基础配置缺失")
        
        self.client = OpenAI(
            api_key=apikey,
            base_url=baseurl,
            timeout=timeout
        )
    
    def think(self, message: list[dict[str, str]], temproture:float = 0) -> str:
        print(f"正在调用{self.model}模型")

        try:
            response = self.client.chat.completions.create(
                model = self.model,
                messages=message,
                temperature=temproture,
                stream=True
            )
            print("模型响应成功")
            collected_content = []
            for chunk in response:
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()
            return "".join(collected_content)
        except Exception as e:
            print(f"调用LLM API时发生错误:{e}")
            return None

try:
    llmClient = HelloAgentLLM(model="deepseek-chat")
    exampleMessage = [
        {"role":"system", "content":"you are a helpful assistant"},
        {"role":"user", "content":"写一个快速排序算法"}
    ]

    print("---调用LLM---")
    responseText = llmClient.think(exampleMessage)
    if responseText:
        print("\n\n--完整模型响应--")
        print(responseText)
except ValueError as e:
    print(e)

---调用LLM---
正在调用deepseek-chat模型
模型响应成功
我来写一个快速排序算法，包含详细注释和示例：

```python
def quick_sort(arr):
    """
    快速排序算法
    
    参数:
    arr: 待排序的列表
    
    返回:
    排序后的列表
    """
    # 递归终止条件：如果数组长度小于等于1，直接返回
    if len(arr) <= 1:
        return arr
    
    # 选择基准元素（这里选择中间元素，也可以选择第一个、最后一个或随机元素）
    pivot = arr[len(arr) // 2]
    
    # 将数组分成三部分：小于基准、等于基准、大于基准
    left = [x for x in arr if x < pivot]      # 小于基准的元素
    middle = [x for x in arr if x == pivot]   # 等于基准的元素
    right = [x for x in arr if x > pivot]     # 大于基准的元素
    
    # 递归排序左右两部分，然后合并
    return quick_sort(left) + middle + quick_sort(right)


def quick_sort_in_place(arr, low=0, high=None):
    """
    原地快速排序（节省内存空间）
    
    参数:
    arr: 待排序的列表
    low: 起始索引
    high: 结束索引
    """
    if high is None:
        high = len(arr) - 1
    
    if low < high:
        # 获取分区点索引
        pi = partition(arr, low, high)
        
        # 递归排序分区点左边的子数组
        quick_sort_in_place(arr, low, pi - 1)
        # 递归排序分区点右边的子数组
        quick_s

In [2]:
from serpapi.serp_api_client import SerpApiClient 

def search(query:str)-> str:
    print(f"正在执行[SerpApi]网页搜索：{query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误:搜索工具apikey有问题"
        params = {
            "engine": "google",
            "q":query,
            "apikey":api_key,
            "gl":"cn",
            "hl":"zh-cn"
        }
        client = SerpApiClient(params)
        results = client.get_dict()
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        return f"没有找到关于“{query}”的信息"
    
    except Exception as e:
        return f"搜索时发生错误:{e}"
    
class ToolExecutor:
    def __init__(self):
        self.tools : dict[str, dict[str, any]] = {}
    def registerTool(self, name:str, description:str, func:callable):
        if name in self.tools:
            print(f"警告：工具{name}已存在，将被覆盖")
        self.tools[name] = {"description": description, "func":func}
        print(f"工具‘{name}’已注册")
    def gettool(self, name:str)-> callable:
        return self.tools.get(name, {}).get("func")
    
    def getAvailableTools(self)->str:
        return "\n".join([
            f"-{name}:{info['description']}"
            for name, info in self.tools.items()
        ])

toolexecutor = ToolExecutor()

search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
toolexecutor.registerTool("Search", search_description, search)

print("\n--可用工具--")
print(toolexecutor.getAvailableTools())

print("\n--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---")
tool_name = "Search"
tool_input = "英伟达最新的GPU型号是什么"

tool_function = toolexecutor.gettool(tool_name)
if tool_function:
    observation = tool_function(tool_input)
    print("--- 观察 (Observation) ---")
    print(observation)
else:
    print(f"错误：未找到名为{tool_name}的工具")



工具‘Search’已注册

--可用工具--
-Search:一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。

--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---
正在执行[SerpApi]网页搜索：英伟达最新的GPU型号是什么
--- 观察 (Observation) ---
[1] 对比各型号NVIDIA GPU
GeForce RTX 5070 Ti, 12 GB, 大型 ; NVIDIA RTX PRO 2000 Blackwell 架构, 8 GB, 中型 ; NVIDIA RTX PRO 3000 Blackwell 架构, 12 GB, 大型 ; 创作大师.

[2] 比较GeForce 系列最新一代显卡和前代显卡| NVIDIA
比较最新一代RTX 30 系列显卡和前代的RTX 20 系列、GTX 10 和900 系列显卡。查看规格、功能、技术支持等内容。

[3] GeForce RTX 50 系列显卡
GeForce RTX™ 50 系列GPU 搭载NVIDIA Blackwell 架构，为游戏玩家和创作者带来全新玩法。RTX 50 系列具备强大的AI 算力，带来升级体验和更逼真的画面。


In [9]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

import re

class ReActAgent:
    def __init__(self, llm_client, tool_executor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []
    
    def run(self, question:str):
        self.history = []
        current_step = 0
        while current_step < self.max_steps:
            current_step += 1
            print(f"---第{current_step}步---")

            tool_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools = tool_desc,
                question = question,
                history = history_str
            )

            messages = [{"role":"user", "content": prompt}]
            response_text = self.llm_client.think(message=messages)

            if not response_text:
                print("错误:LLM 未能成功响应")
                break

            thought, action = self._parse_output(response_text)

            if thought:
                print(f"思考：{thought}")
            if not action:
                print("警告：未能解析出有效Action，流程终止")
                break

            if action.startswith("Finish"):
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"最终答案：{final_answer}")
                return final_answer
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                continue
            print(f"行动：{tool_name}|{tool_input}")

            tool_function = self.tool_executor.gettool(tool_name)
            if not tool_function:
                observation = f"错误：未找到名为'{tool_function}'的工具"
            else:
                observation = tool_function(tool_input)
            print(f"观察：{observation}")

            self.history.append(f"Action:{action}")
            self.history.append(f"Obeservation:{observation}")
        print("已达到最大步数，流程终止")
        return None

    
    def _parse_output(self, text:str):
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action
    
    def _parse_action(self, action_text:str):
        match = re.match(r"(\w*)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None

reactagent = ReActAgent(
    llm_client=llmClient,
    tool_executor=toolexecutor,
    max_steps=3
)

question = "华为最新手机型号及主要卖点"
reactagent.run(question=question)



---第1步---
正在调用deepseek-chat模型
模型响应成功
Thought: 用户询问华为最新手机型号及主要卖点。这是一个需要最新信息的问题，因为手机型号和卖点会随时间更新。我的知识截止日期是2024年7月，可能无法提供最新的信息。为了确保信息的准确性和时效性，我应该使用搜索引擎来查找华为最新发布的手机型号及其主要卖点。

Action: Search[华为最新手机型号 主要卖点 2024]
思考：用户询问华为最新手机型号及主要卖点。这是一个需要最新信息的问题，因为手机型号和卖点会随时间更新。我的知识截止日期是2024年7月，可能无法提供最新的信息。为了确保信息的准确性和时效性，我应该使用搜索引擎来查找华为最新发布的手机型号及其主要卖点。
行动：Search|华为最新手机型号 主要卖点 2024
正在执行[SerpApi]网页搜索：华为最新手机型号 主要卖点 2024
观察：[1] 2024年（11月）19款华为手机推荐
华为mate60pro+是华为最新款的超大杯高端旗舰机型，主要亮点还是那颗自研5G芯片，另外不仅支持卫星通话，还具备双向北斗卫星消息功能，也在后置相机、运行内存上做了升级。

[2] 2024年华为手机哪一款性价比高？华为手机推荐与市场分析
Nova系列：Nova系列主打的优势是性价比，在性能方面紧跟p系列的步伐，针对的是年轻群体，找了很多明星来代言，在华为手机里处于中端水平。 畅享系列：畅享系列的 ...

[3] 华为手机- 华为官网
Mate 系列. 非凡旗舰 · HUAWEI Mate 80 Pro Max 风驰版. ￥8499 起 ; Pura 系列. 先锋影像. HUAWEI Pura 80 Pro+. ￥7999 起 ; Pocket 系列. 美学新篇. HUAWEI Pocket 2 优享版.
---第2步---
正在调用deepseek-chat模型
模型响应成功
Thought: 用户询问华为最新手机型号及主要卖点。根据之前的搜索历史，已经获取了一些信息，但搜索结果中提到了“2024年（11月）19款华为手机推荐”，这可能意味着信息不是最新的。搜索结果中提到了Mate 60 Pro+，但华为官网的搜索结果片段显示有Mate 80 Pro Max和Pura 80 Pro+等型号，这些看起来是更新的产品。为了确保信息的